[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CamiloVga/Curso-IA-Aplicada/blob/main/Semana%2006_Fundamentos_DeepLearning/fundamentos_deep_learning_pytorch.ipynb)

# Fundamentos Deep Learning

## 1. Importar datos y hacer división en train/val/test

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/sample_data/california_housing_train.csv")
df_test = pd.read_csv("/content/sample_data/california_housing_test.csv")

In [ ]:
target = "median_house_value"
X_train = df.drop(target, axis=1)
y_train = df[[target]]
X_test = df.drop(target, axis=1)
y_test = df[[target]]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

## 2. Hacer transformaciones de datos

Este paso es muy importante, de lo contrario, podemos explotar el gradiente!!

In [ ]:
from sklearn.preprocessing import StandardScaler # Pueden usar otras normalizaciones
feature_scaler = StandardScaler()
X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train)
y_val_scaled = target_scaler.transform(y_val)
X_test_scaled =  feature_scaler.transform(X_test)
y_test_scaled = target_scaler.transform(y_test)

## 3. Crear red neuronal y entrenar

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch

# Crear una arquitectura simple muy manual de MLP
class NeuralNetwork(nn.Module):
  def __init__(self, input_size, output_size):
    super(NeuralNetwork, self).__init__()
    # Definimos las capas de forma manual
    self.layer1 = nn.Linear(input_size, 3)
    self.layer2 = nn.Linear(3,3)
    self.layer3 = nn.Linear(3,3)
    self.layer4 = nn.Linear(3,output_size)
  def forward(self, x):
    out1 = F.relu(self.layer1(x))
    out2 = F.relu(self.layer2(out1))
    out3 = F.relu(self.layer3(out2))
    out4 = self.layer4(out3)
    return out4

# Instanciar el modelo
model = NeuralNetwork(input_size=8, output_size=1)

# Definir funcion de pérdida y optimizador
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)


# Crear tensores en pytorch con los datos
X_train_tensor = torch.Tensor(X_train_scaled)
y_train_tensor = torch.Tensor(y_train_scaled)
X_val_tensor = torch.Tensor(X_val_scaled)
y_val_tensor = torch.Tensor(y_val_scaled)

# Realizar loop de entrenamiento
# Ojo, notemos que estamos haciendo Batch Gradient Descent por que actualizamos
# los pesos con el conjunto de datos completo.
for epoch in range(5):
  predictions = model(X_train_tensor)
  loss = criterion(predictions, y_train_tensor)

  # Backward pass y optimizacion
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  print(f"Epoca {epoch+1}: train_loss", loss.item())

Epoca 1: train_loss 1.1737157106399536
Epoca 2: train_loss 1.172805666923523
Epoca 3: train_loss 1.171900749206543
Epoca 4: train_loss 1.1710007190704346
Epoca 5: train_loss 1.1701048612594604


Noten que no estamos siguiendo la pérdida en el conjunto de validación.

## 4. Generalización de redes neuronales y entrenamiento


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch

# Crear la arquitectura MLP (es la que vimos en clase)
class MLP(nn.Module):
    def __init__(self, input_size, hidden_layers, hidden_neurons, output_size):
        super(MLP, self).__init__()
        layers = []
        # Primera capa
        layers.append(nn.Linear(input_size, hidden_neurons))
        layers.append(nn.ReLU())
        # Capas ocultas
        for i in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_neurons, hidden_neurons))
            layers.append(nn.ReLU())
        # Capa de salida (sin función de activación)
        layers.append(nn.Linear(hidden_neurons, output_size))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

torch.manual_seed(45) # Importante para reproducibilidad

# Hiperparámetros
hidden_layers = 4
hidden_neurons = 256
learning_rate = 0.001
batch_size = 64
num_epochs = 100

# Instanciar el modelo
model = MLP(input_size=8,
            hidden_layers=hidden_layers,
            hidden_neurons=hidden_neurons,
            output_size=1)

# Definir funcion de pérdida y optimizador
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate) # Usamos un optimizador que converge más rapido, como AdamW

# Crear dataset de torch para hacer más sencillo la partición en batches
# para realizar el algoritmo mini-batch gradient descent
from torch.utils.data import TensorDataset, DataLoader

# Crear datasets
train_dataset = TensorDataset(torch.Tensor(X_train_scaled), torch.Tensor(y_train_scaled))
val_dataset = TensorDataset(torch.Tensor(X_val_scaled), torch.Tensor(y_val_scaled))
test_dataset = TensorDataset(torch.Tensor(X_test_scaled), torch.Tensor(y_test_scaled))

# Crear dataloaders, estos permiten hacer particiones y manipulaciones más sencillas sobre batches
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Loop de entrenamiento
for epoch in range(num_epochs):
  model.train()
  train_loss = 0
  for batch_x, batch_y in train_loader:
    predictions = model(batch_x)
    loss = criterion(predictions, batch_y)

    # Backward pass y optimizacion
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_loss = train_loss + (loss.item() * batch_x.size(0))
  train_loss = train_loss / len(train_loader.dataset)

  model.eval()
  val_loss = 0
  with torch.no_grad():
      for batch_x, batch_y in val_loader:
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        val_loss = val_loss + (loss.item()*batch_x.size(0))
  val_loss = val_loss / len(val_loader.dataset)
  print(f"Epoca {epoch+1}: train_loss={ train_loss },val_loss={ val_loss }")

Epoca 1: train_loss=0.3589868079213535,val_loss=0.2604232985131881
Epoca 2: train_loss=0.26769562360118415,val_loss=0.2657970483864055
Epoca 3: train_loss=0.2485207264213001,val_loss=0.24351007517646342
Epoca 4: train_loss=0.23434144426794612,val_loss=0.23051427350324744
Epoca 5: train_loss=0.22653130587409526,val_loss=0.22052771284299738
Epoca 6: train_loss=0.2189896740282283,val_loss=0.23182700339485618
Epoca 7: train_loss=0.2136960747312097,val_loss=0.22886832328403697
Epoca 8: train_loss=0.21252520080874948,val_loss=0.21865874339552488
Epoca 9: train_loss=0.20526010923525867,val_loss=0.21614987702930674
Epoca 10: train_loss=0.20460585548597224,val_loss=0.21445761890972362
Epoca 11: train_loss=0.19963500135085163,val_loss=0.2116601456263486
Epoca 12: train_loss=0.1958647325810264,val_loss=0.22231652375529795
Epoca 13: train_loss=0.19627484886085286,val_loss=0.20267270726316117
Epoca 14: train_loss=0.19345774995930054,val_loss=0.2097262724357493
Epoca 15: train_loss=0.189415577194269

## Y la GPU?

Hasta ahora no hemos necesitado la GPU, pero el entrenamiento se hace mucho más eficiente cuando trabajamos con esta. Recuerda cambiar el runtime de colab!

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch

# Verificar si hay GPU disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

# Si hay GPU, mostrar información
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

class MLP(nn.Module):
    def __init__(self, input_size, hidden_layers, hidden_neurons, output_size):
        super(MLP, self).__init__()
        layers = []
        layers.append(nn.Linear(input_size, hidden_neurons))
        layers.append(nn.ReLU())
        for i in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_neurons, hidden_neurons))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_neurons, output_size))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

torch.manual_seed(45)
if torch.cuda.is_available():
    torch.cuda.manual_seed(45)  # También fijar seed en GPU

# Hiperparámetros
hidden_layers = 4
hidden_neurons = 256
learning_rate = 0.001
batch_size = 64
num_epochs = 100

# Instanciar el modelo y moverlo a GPU
model = MLP(input_size=8,
            hidden_layers=hidden_layers,
            hidden_neurons=hidden_neurons,
            output_size=1).to(device)  # ← MOVER A GPU

# Definir función de pérdida y optimizador
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Crear datasets (manteniendo los datos en CPU por ahora)
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(torch.Tensor(X_train_scaled), torch.Tensor(y_train_scaled))
val_dataset = TensorDataset(torch.Tensor(X_val_scaled), torch.Tensor(y_val_scaled))
test_dataset = TensorDataset(torch.Tensor(X_test_scaled), torch.Tensor(y_test_scaled))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Loop de entrenamiento
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        # Mover datos a GPU
        batch_x = batch_x.to(device)  # ← MOVER BATCH A GPU
        batch_y = batch_y.to(device)  # ← MOVER BATCH A GPU

        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss = train_loss + (loss.item() * batch_x.size(0))
    train_loss = train_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            # Mover datos a GPU
            batch_x = batch_x.to(device)  # ← MOVER BATCH A GPU
            batch_y = batch_y.to(device)  # ← MOVER BATCH A GPU

            predictions = model(batch_x)
            loss = criterion(predictions, batch_y)
            val_loss = val_loss + (loss.item() * batch_x.size(0))
    val_loss = val_loss / len(val_loader.dataset)
    print(f"Epoca {epoch+1}: train_loss={train_loss:.6f}, val_loss={val_loss:.6f}")

Usando dispositivo: cuda
GPU: Tesla T4
Memoria GPU: 15.83 GB
Epoca 1: train_loss=0.359066, val_loss=0.260358
Epoca 2: train_loss=0.267980, val_loss=0.266908
Epoca 3: train_loss=0.249608, val_loss=0.241430
Epoca 4: train_loss=0.234378, val_loss=0.229381
Epoca 5: train_loss=0.226576, val_loss=0.221366
Epoca 6: train_loss=0.220220, val_loss=0.230192
Epoca 7: train_loss=0.212834, val_loss=0.228574
Epoca 8: train_loss=0.212431, val_loss=0.217818
Epoca 9: train_loss=0.205602, val_loss=0.217125
Epoca 10: train_loss=0.205377, val_loss=0.215798
Epoca 11: train_loss=0.199744, val_loss=0.209952
Epoca 12: train_loss=0.195899, val_loss=0.224591
Epoca 13: train_loss=0.198662, val_loss=0.204684
Epoca 14: train_loss=0.193445, val_loss=0.212404
Epoca 15: train_loss=0.188562, val_loss=0.210185
Epoca 16: train_loss=0.187972, val_loss=0.209983
Epoca 17: train_loss=0.182791, val_loss=0.211109
Epoca 18: train_loss=0.185276, val_loss=0.195875
Epoca 19: train_loss=0.180871, val_loss=0.206330
Epoca 20: train_l